# Brewline Coffee Co. — Customer & Sales Analysis

**Step 1: Data Preparation, Modeling & Exploratory Data Analysis (Python)**

This notebook cleans a year of transaction data across 3 store locations, engineers
a few analysis-ready fields, and explores revenue, customer, and product patterns
before handing off to SQL (Step 2) and Power BI / the dashboard (Step 3).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

## 1. Load raw data

In [ ]:
customers = pd.read_csv("../data/customers_raw.csv")
orders = pd.read_csv("../data/orders_raw.csv")

print("customers:", customers.shape)
print("orders:", orders.shape)
customers.head()

In [ ]:
orders.head()

## 2. Inspect data quality

In [ ]:
print("Missing values — customers:")
print(customers.isna().sum())
print()
print("Missing values — orders:")
print(orders.isna().sum())
print()
print("Duplicate customer rows:", customers.duplicated(subset='customer_id').sum())
print("Exact duplicate order rows:", orders.duplicated().sum())
print("Orders with quantity <= 0:", (orders['quantity'] <= 0).sum())
print()
print("Distinct store_location values (note casing/whitespace issues):")
print(orders['store_location'].unique())

## 3. Clean the data

Issues found and how we handle them:
- **Duplicate customers / orders** → drop exact duplicates
- **Missing `age`** → impute with median age
- **Missing `gender`** → label as `"Not specified"` rather than dropping the row
- **Inconsistent `store_location` casing/whitespace** → strip + title-case
- **`quantity <= 0`** (POS entry errors) → drop those rows
- **Missing `satisfaction_rating`** → left as-is (not every customer rates their order);
  handled with `NaN`-aware aggregation later, not imputed

In [ ]:
# --- Clean customers ---
customers = customers.drop_duplicates(subset="customer_id").copy()
customers["age"] = customers["age"].fillna(customers["age"].median()).astype(int)
customers["gender"] = customers["gender"].fillna("Not specified")
customers["signup_date"] = pd.to_datetime(customers["signup_date"])

def age_group(a):
    if a < 25: return "18-24"
    if a < 35: return "25-34"
    if a < 45: return "35-44"
    if a < 55: return "45-54"
    return "55+"

customers["age_group"] = customers["age"].apply(age_group)
customers.head()

In [ ]:
# --- Clean orders ---
orders = orders.drop_duplicates().copy()
orders["store_location"] = orders["store_location"].str.strip().str.title()
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders = orders[orders["quantity"] > 0].copy()

orders["total_amount"] = (orders["quantity"] * orders["unit_price"]).round(2)
orders["month"] = orders["order_date"].dt.month_name()
orders["day_of_week"] = orders["order_date"].dt.day_name()

print("Cleaned store_location values:", orders["store_location"].unique())
orders.head()

In [ ]:
# --- Merge into one analysis-ready table ---
df = orders.merge(customers, on="customer_id", how="left")
print("Final shape:", df.shape)
df.head()

## 4. Exploratory Data Analysis

### 4.1 Monthly revenue trend

In [ ]:
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
monthly = df.groupby("month")["total_amount"].sum().reindex(month_order)

plt.figure(figsize=(9,4))
monthly.plot(kind="line", marker="o")
plt.title("Monthly Revenue — 2025")
plt.ylabel("Revenue ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.2 Revenue by category and store

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

df.groupby("category")["total_amount"].sum().sort_values().plot(
    kind="barh", ax=axes[0], color="#7B5B41")
axes[0].set_title("Revenue by Category")
axes[0].set_xlabel("Revenue ($)")

df.groupby("store_location")["total_amount"].sum().sort_values().plot(
    kind="barh", ax=axes[1], color="#4C7A6D")
axes[1].set_title("Revenue by Store")
axes[1].set_xlabel("Revenue ($)")

plt.tight_layout()
plt.show()

### 4.3 Who are our customers? Revenue by age group, loyalty status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

df.groupby("age_group")["total_amount"].sum().reindex(
    ["18-24","25-34","35-44","45-54","55+"]).plot(kind="bar", ax=axes[0], color="#B08968")
axes[0].set_title("Revenue by Age Group")
axes[0].set_ylabel("Revenue ($)")

df.groupby("loyalty_member")["total_amount"].mean().plot(kind="bar", ax=axes[1], color="#8C6A56")
axes[1].set_title("Average Order Value: Loyalty vs. Non-Member")
axes[1].set_xticklabels(["Non-member", "Loyalty member"], rotation=0)
axes[1].set_ylabel("Avg order value ($)")

plt.tight_layout()
plt.show()

### 4.4 Top-selling items

In [ ]:
top_items = df.groupby("item")["total_amount"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(8,4))
top_items.sort_values().plot(kind="barh", color="#A9746E")
plt.title("Top 10 Items by Revenue")
plt.xlabel("Revenue ($)")
plt.tight_layout()
plt.show()

## 5. Export cleaned data

These cleaned tables feed:
- **Step 2 (SQL):** loaded into a database for business-question queries
- **Step 3 (Dashboard):** aggregated into a summary the dashboard visualizes

In [ ]:
customers.to_csv("../data/customers_clean.csv", index=False)
orders.to_csv("../data/orders_clean.csv", index=False)
df.to_csv("../data/full_clean.csv", index=False)
print("Saved cleaned CSVs to ../data/")

## 6. Key takeaways

- **Downtown** is the top-performing store by revenue, followed by Riverside and University Ave.
- **Espresso Drinks** drive the largest share of revenue, led by Flat White and Cappuccino.
- **Customers 55+** generate the most total revenue despite not being the largest age group by order count — worth digging into in Step 2 (SQL) as a loyalty/frequency question.
- Loyalty members and non-members have a nearly identical average order value — loyalty membership doesn't appear to change *how much* someone spends per visit, only (potentially) how *often* they visit, which we investigate next with SQL.

## 7. RFM Segmentation (extending the analysis)

The loyalty comparison above only splits customers two ways (member vs. non-member) and looks at averages,
which is how it missed a real difference in behavior. RFM goes further: it scores *every* customer on
**Recency** (days since their last order), **Frequency** (how many orders they've placed), and **Monetary**
value (how much they've spent total), then buckets them into segments — Champions, At Risk, Lost, and so on —
regardless of loyalty status. It's a sharper version of the same question: which customers should Brewline
actually be worried about losing?

In [ ]:
# --- Compute RFM metrics per customer ---
reference_date = df["order_date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("customer_id").agg(
    recency=("order_date", lambda x: (reference_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("total_amount", "sum"),
).reset_index()

rfm.head()

In [ ]:
# --- Score each metric 1 (worst) to 5 (best) using quintiles ---
# .rank(method="first") breaks ties so qcut always produces 5 even-ish bins
rfm["R_score"] = pd.qcut(rfm["recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

rfm.head()

### 7.1 Turning scores into segments

Raw scores are hard to act on, so we map score combinations to plain-language segments — a standard,
simplified version of the usual RFM segment rules:

In [ ]:
def rfm_segment(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Champions"
    elif row["R_score"] >= 4 and row["F_score"] >= 3:
        return "Loyal Customers"
    elif row["R_score"] >= 4 and row["F_score"] <= 2:
        return "New / Promising"
    elif row["R_score"] <= 2 and row["F_score"] >= 4:
        return "At Risk"
    elif row["R_score"] <= 2 and row["F_score"] <= 2:
        return "Lost / Hibernating"
    else:
        return "Needs Attention"

rfm["segment"] = rfm.apply(rfm_segment, axis=1)
rfm["segment"].value_counts()

In [ ]:
segment_order = ["Champions", "Loyal Customers", "New / Promising",
                  "Needs Attention", "At Risk", "Lost / Hibernating"]

plt.figure(figsize=(9, 4))
rfm["segment"].value_counts().reindex(segment_order).plot(kind="bar", color="#6F4E37")
plt.title("Customers by RFM Segment")
plt.ylabel("Customers")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 7.2 Does loyalty membership track with RFM health?

Section 4.3 found loyalty members and non-members spend about the same per order. RFM lets us ask a more
specific version of that question: are loyalty members actually more concentrated in the *healthy* segments
(Champions, Loyal), or are they spread evenly across the risk spectrum too?

In [ ]:
loyalty_by_customer = df.groupby("customer_id")["loyalty_member"].first().reset_index()
rfm = rfm.merge(loyalty_by_customer, on="customer_id", how="left")

pd.crosstab(rfm["segment"], rfm["loyalty_member"], normalize="index").reindex(segment_order).round(2)

In [ ]:
# --- Export for the SQL / Power BI layer ---
rfm.to_csv("../data/customer_segments.csv", index=False)
print("Saved RFM segments to ../data/customer_segments.csv")

### 7.3 RFM takeaways

- **206 customers (31.7%) are At Risk or Lost/Hibernating** — high past value or frequency, but no recent
  orders. That's roughly a third of the customer base that a win-back campaign (email, a loyalty nudge,
  a "we miss you" offer) could realistically target.
- **Champions are only 17.2% of customers but generate 24.8% of total revenue** — a concrete number for
  "protect your best customers first" if Brewline has to prioritize where to spend retention effort.
- **Loyalty membership does skew toward the healthy segments, just not by much:** 51% of Champions and 49%
  of Loyal Customers are loyalty members, versus 42% of At Risk and 38% of Lost/Hibernating customers
  (baseline loyalty membership rate across all customers is 44.8%). So the program isn't doing *nothing* —
  it's mildly associated with staying healthy — but the effect is small enough that it doesn't show up when
  you only compare group averages, which is exactly why the aggregate AOV/frequency comparison in Section 4.3
  looked like a flat line. Segmenting individual customers surfaces a signal that averaging washes out.
- **Next step if this were a real business:** target the At Risk segment specifically with a loyalty-program
  invite or reactivation offer, then re-run this segmentation next quarter to see if membership rate among
  "healthy" segments moves beyond the current mild skew.